# 소비기한 추출 — [DAT] 캐치캐치

상품 뒷면 이미지에서 소비기한을 추출해 `submission.csv` 를 만든다.

**설계 요지.** 이 과제의 병목은 인식이 아니라 **선별**과 **속도**다.
이미지의 71%에 날짜 모양 오답(품목보고번호·바코드·전화번호·로트코드)이 있고,
글자 밀도가 높아(1000px 기준 연결성분 중앙값 88개) 검출된 텍스트를 전부 인식하면
어떤 모델을 써도 예산을 초과한다. 그래서 **검출과 인식 사이에 필터를 넣어**
인식 크롭을 2~3개로 줄이고, 그 뒤 규칙 캐스케이드로 소비기한 하나를 고른다.

파이프라인은 **anytime 설계**다 — 시작 직후부터 유효한 CSV가 디스크에 존재하고
이후 장마다 개선된다. 어디서 중단돼도 산출물이 남는다.

세부 근거는 `docs/PIPELINE.md`, 인용 문헌은 `docs/선행연구.md`.

In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

## 1. 환경 고정

`onnxruntime` 과 OpenMP는 **import 시점에** 스레드 수를 읽는다. 그래서 스레드
설정이 import보다 먼저 와야 한다 (측정 재현성의 전제이기도 하다 —
Mytkowicz et al., ASPLOS 2009).

In [ ]:
import sys, time, csv, re
from pathlib import Path

T0 = time.time()

# --- 스레드 고정: 반드시 cv2 / onnxruntime import 보다 먼저 ---
THREADS = 4                                   # 채점 환경 = Standard 4-Core vCPU
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(THREADS)

# --- 루트 리졸버 ---
# 채점기가 저장소 밖에서 nbconvert 를 부르면 cwd 가 저장소 루트가 아닐 수 있다.
# itda_ocr/ 를 담은 디렉터리를 위로 훑어 찾는다.
def _find_root():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "itda_ocr" / "pipeline.py").exists():
            return base
    raise RuntimeError("itda_ocr 패키지를 찾지 못했습니다. 저장소 루트에서 실행해 주세요.")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from itda_ocr.pipeline import Config, iter_images, run

# 2400초 제한. 여유를 두고 마감을 잡아 저장·검증 시간을 남긴다.
DEADLINE = T0 + 2100
print(f"root={ROOT}  threads={THREADS}  budget={DEADLINE - T0:.0f}s")

## 2. 설정

임계값을 코드에 흩뿌리지 않고 한 곳에 모았다. 각 값의 근거는 주석에 있다.

In [ ]:
CFG = Config(
    # DCT 단계 축소 디코딩. ≥4MP 코호트(19.8%)가 318ms → 103ms.
    # 디코딩만으로 예산의 28%를 쓰므로 최적화가 아니라 필수 요건이다.
    draft_to=720,
    # 검출 입력 상한(long side). ExpDate 665장 실측 검출 recall:
    # 480px 75.2% / 640px 83.6% / 960px 86.8%. 480→640이 +8.4pp를 ~23ms에 산다.
    det_side=640,
    # 인식 배치 크기. 이만큼씩 읽고 유효 날짜가 나오면 멈춘다.
    top_k=2,
    # 조기 종료가 없을 때의 크롭 수 상한.
    # ExpDate 실측이 이 값을 만들었다: 박스 필터는 정답 박스를 하나도 버리지
    # 않는데(손실 0.0%), 고정 K=2에서는 정답이 상위 2위 안에 드는 경우가 35.3%뿐이라
    # 실패의 57%가 '후보 아예 없음'이었다. 병목은 필터가 아니라 순위다.
    # recall@K: 1→24.0% · 3→44.3% · 5→54.7% · 8→65.0% · 12→69.7%
    max_k=9,
    threads=THREADS,
    # 인쇄물에 일자가 없으면 정답도 NONE이다(확정 규칙). 따라서 빈 칸을 지어내지
    # 않는다. 실측도 같은 방향 — 보정은 ExpDate 665장에서 50점 만점에 1.02점을 깎았다.
    impute_missing=False,
    box_thresh=0.5,
    unclip_ratio=1.6,
    per_image_budget=0.15,
    flush_every=50,
    # Track B — 날짜 전용 검출기(NanoDet-Plus-m @480, ONNX 5.6MB).
    #   weights/date_detector_ema.onnx 가 있으면 detect() 가 RapidOCR 대신 이걸 쓴다.
    #   ExpDate eval 665장 end-to-end: 36.85 -> 39.71 / 50 (검출 recall 82.9% -> 97.6%).
    #   det_side 는 이때 무시된다 (NanoDet 내부 480 고정).
    nanodet_onnx=(lambda p: str(p) if p.exists() else None)(ROOT / "weights" / "date_detector_ema.onnx"),
    nanodet_score_thr=0.05,
    # NanoDet 회귀 박스는 글자에 딱 붙어 나온다(RapidOCR 은 unclip_ratio=1.6 으로
    # 이미 부풀려서 준다). 그대로 자르면 크롭 경계에서 끝 글자가 잘려
    # '2021.08.04' 가 '2021.08.0' 이 된다 — 665장 중 88장이 이렇게 퇴행했다.
    # 0.00~0.30 스윕에서 0.06~0.15 가 평탄해 그 중앙을 골랐다 (39.71 -> 41.41 / 50).
    nanodet_expand=0.10,
)

from itda_ocr.engine import Engine
_t = time.time()
ENGINE = Engine(det_side=CFG.det_side, threads=CFG.threads,
                box_thresh=CFG.box_thresh, unclip_ratio=CFG.unclip_ratio,
                nanodet_onnx=CFG.nanodet_onnx,
                nanodet_score_thr=CFG.nanodet_score_thr,
                nanodet_expand=CFG.nanodet_expand)
# 모델 로드는 1회 비용이므로 장당 비용과 분리해 보고한다 (Georges et al., OOPSLA 2007).
_det_name = "NanoDet 날짜검출기" if CFG.nanodet_onnx else "RapidOCR 범용검출"
print(f"engine ready in {time.time() - _t:.2f}s  |  검출: {_det_name}  "
      f"(가중치는 rapidocr-onnxruntime wheel 내장 — 다운로드 없음)")

## 3~4. 실행

`run()` 은 첫 동작으로 **전부 `NONE` 인 유효 CSV를 저장**한다. nbconvert 타임아웃은
커널을 강제 종료해서 `finally` 가 돌지 않을 수 있으므로, 이 선기록이 빈 결과 파일에
대한 유일한 구조적 보증이다.

이후 장마다 결과를 덮어쓰며, 남은 예산이 부족해지면 `top_k` 를 낮추고(단계 하향)
그래도 모자라면 남은 이미지를 `NONE` 으로 둔 채 종료한다 — 예산을 쉬운 입력과
어려운 입력에 고르지 않게 쓰는 budgeted batch 설계(Huang et al., ICLR 2018).

In [ ]:
N = len(iter_images(INPUT_DIR))
print(f"{N}장  장당 예산 {(DEADLINE - time.time()) / max(N, 1) * 1000:.0f} ms")

RESULT = run(INPUT_DIR, OUTPUT_PATH, CFG, engine=ENGINE, deadline=DEADLINE)

## 6. 스키마 검증

**이 셀은 어떤 경우에도 raise하지 않는다.** 형식이 어긋나면 안전값으로 되돌리고
경고만 남긴다 — 예외를 던지면 정량 60점이 통째로 0점이 된다.

In [ ]:
PAT = {"year": re.compile(r"^\d{4}$"), "month": re.compile(r"^\d{2}$"),
       "day": re.compile(r"^\d{2}$"),
       "final_date": re.compile(r"^\d{4}-\d{2}-\d{2}$")}
FIELDS = ["image_id", "year", "month", "day", "final_date"]

rows, repaired = RESULT["rows"], 0
for r in rows:
    for key, pat in PAT.items():
        v = r.get(key, "NONE")
        if not isinstance(v, str):
            v = str(v)
        if v != "NONE" and not pat.match(v):
            v, repaired = "NONE", repaired + 1
        r[key] = v
    # 불변식: final_date 와 y/m/d 는 서로 모순되면 안 된다.
    ymd = (r["year"], r["month"], r["day"])
    if r["final_date"] != "NONE" and r["final_date"] != "-".join(ymd):
        r["final_date"], repaired = "NONE", repaired + 1

with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

if len(rows) != N:
    print(f"경고: 행 수 불일치 ({len(rows)} != {N})")
print(f"saved {len(rows)} rows to {OUTPUT_PATH}" + (f"  (복구 {repaired}건)" if repaired else ""))

## 7. 실행 리포트

비용이 0이고, 채점기는 실행된 노트북을 보관한다 — 속도 10점에 대한 우리 쪽 근거가 된다.

In [ ]:
el = RESULT["elapsed"]
print(f"이미지        {RESULT['n']}장")
print(f"경과          {el:.1f}s  (모델 로드 제외)")
print(f"장당          {RESULT['ms_per_image']:.0f} ms")
print(f"커버리지      {RESULT['coverage']:.1%}   (날짜를 낸 비율)")
print(f"단계 하향     {RESULT['degraded']}장")
print(f"예산 초과 생략 {RESULT['skipped']}장")
print(f"전체 벽시계   {time.time() - T0:.1f}s / 2400s")